## End-to-End NLP & Deep Learning Document Classification

**Author:** Data Science Intern (@Prpandav)  
**Dataset:** Inshorts News Dataset (300k+ Articles)  
**Objectives:**
1. Comprehensive text preprocessing, data sanitization, and category normalization.
2. Advanced NLP Exploratory Data Analysis (EDA): distributions, sequence lengths, bigrams, and word clouds.
3. Machine Learning Modeling: Multinomial Naive Bayes, Logistic Regression, and Calibrated LinearSVC.
4. **Deep Learning Modeling:** Multi-Filter 1D Convolutional Neural Network (PyTorch TextCNN) with learned embeddings.
5. Deployment: Model export for the interactive Streamlit Web Application.

In [ ]:
import os
import re
import ast
import time
import json
import joblib
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Visual configurations
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['font.size'] = 11
print('Libraries imported successfully!')

### Task 1: Dataset Loading & Structure Inspection

In [ ]:
# Load the Inshorts news dataset
file_path = r'../Data/inshort_news_dataset.csv'
print('Loading dataset...')
df = pd.read_csv(file_path, nrows=80000, usecols=['Headline', 'Content', 'News Categories'])

print(f'Total Articles Loaded: {len(df):,}')
print(f'Features: {df.columns.tolist()}')
display(df.head(3))

### Task 2: Data Cleaning, Category Normalization & Feature Fusion
1. **Category Parsing:** Inshorts categories are stored as Python list strings (e.g. `"['politics', 'national']"`). We parse these into clean, atomic topic categories.
2. **Feature Fusion:** The `Headline` contains the highest-density summary keywords. We merge `Headline + " - " + Content` to eliminate ambiguity.

In [ ]:
# 1. Clean Category Extraction
def extract_primary_category(val):
    if not isinstance(val, str):
        return None
    try:
        cats = ast.literal_eval(val)
        if isinstance(cats, list) and len(cats) > 0:
            return cats[0].strip().lower()
    except Exception:
        pass
    val = val.strip("[]'\" ").lower()
    return val if val else None

df['category'] = df['News Categories'].apply(extract_primary_category)

# Filter top 10 distinct categories
TARGET_CATEGORIES = [
    'business', 'world', 'entertainment', 'politics', 
    'automobile', 'national', 'science', 'sports', 
    'education', 'technology'
]
nlp_df = df[df['category'].isin(TARGET_CATEGORIES)].copy()
print(f'Articles retained across Top 10 Target Categories: {len(nlp_df):,}')

# 2. Text Normalization Function
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 3. Merging Headline and Content
nlp_df['full_text'] = (nlp_df['Headline'].fillna('') + ' - ' + nlp_df['Content'].fillna('')).apply(clean_text)
print('Feature fusion complete! Sample:')
print(nlp_df[['category', 'full_text']].head(2))

### Task 3: Advanced NLP Exploratory Data Analysis (EDA)

In [ ]:
# 1. Category Distribution Plot
plt.figure(figsize=(10, 5))
cat_counts = nlp_df['category'].value_counts()
sns.barplot(x=cat_counts.values, y=cat_counts.index, palette='mako')
plt.title('Top 10 News Categories Distribution', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Number of Articles')
plt.tight_layout()
plt.show()

# 2. Text Length Distribution
nlp_df['word_count'] = nlp_df['full_text'].apply(lambda x: len(x.split()))
plt.figure(figsize=(10, 4))
sns.histplot(nlp_df['word_count'], bins=50, kde=True, color='teal')
plt.title('Distribution of Article Word Counts', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Word Count (Tokens)')
plt.tight_layout()
plt.show()

# 3. Word Cloud Visualization
sample_corpus = ' '.join(nlp_df['full_text'].sample(n=5000, random_state=42).tolist())
wordcloud = WordCloud(width=1000, height=500, background_color='white', colormap='viridis', max_words=120).generate(sample_corpus)
plt.figure(figsize=(11, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Top Vocabulary', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

### Task 4: Feature Engineering & Train-Test Splitting

In [ ]:
X = nlp_df['full_text']
y_text = nlp_df['category']

# Encode categorical labels to integers
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)
class_names = [c.capitalize() for c in label_encoder.classes_]

# 80% Train, 20% Test with Stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Sublinear TF-IDF Vectorizer with N-Gram range (1, 2)
tfidf_vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english'
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f'Train samples: {X_train_tfidf.shape[0]}, Features: {X_train_tfidf.shape[1]}')
print(f'Test samples:  {X_test_tfidf.shape[0]}')

### Task 5: Traditional Machine Learning Models (Naive Bayes, Logistic Regression & LinearSVC)

In [ ]:
# 1. Multinomial Naive Bayes (Baseline)
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
y_pred_nb = nb_model.predict(X_test_tfidf)
print(f'Multinomial Naive Bayes Accuracy: {accuracy_score(y_test, y_pred_nb)*100:.2f}%')

# 2. Logistic Regression
lr_model = LogisticRegression(max_iter=1000, C=1.5, random_state=42)
lr_model.fit(X_train_tfidf, y_train)
y_pred_lr = lr_model.predict(X_test_tfidf)
print(f'Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr)*100:.2f}%')

# 3. Calibrated LinearSVC (Fast, Calibrated Probabilities)
base_svc = LinearSVC(C=1.0, random_state=42, dual=False)
calibrated_svc = CalibratedClassifierCV(estimator=base_svc, cv=3)
calibrated_svc.fit(X_train_tfidf, y_train)
y_pred_svc = calibrated_svc.predict(X_test_tfidf)
svc_acc = accuracy_score(y_test, y_pred_svc)
print(f'Calibrated LinearSVC Accuracy: {svc_acc*100:.2f}%\n')

print('Classification Report (Calibrated LinearSVC):')
print(classification_report(y_test, y_pred_svc, target_names=class_names))

### Task 6: Deep Learning Model — PyTorch TextCNN (1D Multi-Filter CNN)
Traditional TF-IDF models ignore word order. Our **PyTorch TextCNN** uses 1D Convolutions with filter banks of sizes 2, 3, and 4 to capture local semantic phrases across 128-dimensional learned embeddings.

In [ ]:
# Build Vocabulary for PyTorch TextCNN
token_counts = Counter()
for text in X_train:
    token_counts.update(text.split())

VOCAB_SIZE = 25000
MAX_LEN = 100
vocab = {'<PAD>': 0, '<UNK>': 1}
for word, _ in token_counts.most_common(VOCAB_SIZE - 2):
    vocab[word] = len(vocab)

def text_to_seq(text):
    tokens = text.split()[:MAX_LEN]
    seq = [vocab.get(w, 1) for w in tokens]
    if len(seq) < MAX_LEN:
        seq += [0] * (MAX_LEN - len(seq))
    return seq

X_train_seq = np.array([text_to_seq(t) for t in X_train], dtype=np.int64)
X_test_seq = np.array([text_to_seq(t) for t in X_test], dtype=np.int64)

class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(TextDataset(X_train_seq, y_train), batch_size=128, shuffle=True)
test_loader = DataLoader(TextDataset(X_test_seq, y_test), batch_size=256, shuffle=False)

# TextCNN Architecture
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_classes=10, num_filters=64, filter_sizes=[2, 3, 4]):
        super(TextCNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])
        self.dropout = nn.Dropout(0.40)
        self.fc = nn.Linear(num_filters * len(filter_sizes), num_classes)
        
    def forward(self, x):
        emb = self.embedding(x).transpose(1, 2)  # [batch, embed_dim, seq_len]
        conv_outs = [torch.relu(conv(emb)) for conv in self.convs]
        pooled = [torch.max(co, dim=2)[0] for co in conv_outs]   # Global max pool
        cat = self.dropout(torch.cat(pooled, dim=1))
        return self.fc(cat)

device = torch.device('cpu')
dl_model = TextCNN(vocab_size=len(vocab), embed_dim=128, num_classes=len(class_names)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(dl_model.parameters(), lr=0.003, weight_decay=1e-4)
print('PyTorch TextCNN instantiated successfully!')

In [ ]:
# Train PyTorch TextCNN
EPOCHS = 3
print('Training PyTorch TextCNN...')
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    dl_model.train()
    total_loss = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = dl_model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    # Evaluate
    dl_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for bx, by in test_loader:
            bx, by = bx.to(device), by.to(device)
            preds = dl_model(bx).argmax(dim=1)
            correct += (preds == by).sum().item()
            total += len(by)
            
    acc = correct / total
    print(f'Epoch {epoch}/{EPOCHS} [{time.time()-t0:.1f}s] - Train Loss: {total_loss/len(train_loader):.4f} - Test Accuracy: {acc*100:.2f}%')

### Task 7: Confusion Matrix Evaluation

In [ ]:
# Generate predictions on test set
dl_model.eval()
all_preds = []
with torch.no_grad():
    for bx, by in test_loader:
        bx = bx.to(device)
        all_preds.extend(dl_model(bx).argmax(dim=1).cpu().numpy())

cm = confusion_matrix(y_test, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix: PyTorch TextCNN (Test Set)', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Predicted Category', fontweight='bold')
plt.ylabel('Actual Category', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Task 8: Live Inference Function & Pipeline Serialization

In [ ]:
# Live Prediction Function
def predict_article(headline, content, model_type='dl'):
    full = (headline + ' - ' + content).lower()
    cl = clean_text(full)
    
    if model_type == 'dl':
        seq = torch.tensor([text_to_seq(cl)], dtype=torch.long)
        with torch.no_grad():
            logits = dl_model(seq)
            probs = torch.softmax(logits, dim=1).numpy()[0]
        pred_idx = np.argmax(probs)
        return class_names[pred_idx], probs[pred_idx]
    else:
        vec = tfidf_vectorizer.transform([cl])
        probs = calibrated_svc.predict_proba(vec)[0]
        pred_idx = np.argmax(probs)
        return class_names[pred_idx], probs[pred_idx]

# Test Predictions
test_sample = {
    'headline': 'ISRO prepares advanced satellite launch for lunar observation',
    'content': 'The space agency successfully assembled the rocket stage payload at the spaceport.'
}

category, conf = predict_article(test_sample['headline'], test_sample['content'], model_type='dl')
print(f"Predicted Category: {category} (Confidence: {conf*100:.2f}%)")